In [1]:
# 08_robustness_inter_model_consistency.ipynb
# Inter-model stability of the counterfactual recommendations. For each sex, three
# DiCE-compatible tree ensembles (XGBoost, CatBoost, LightGBM) are fit on the full data;
# for each of 10 sampled high-risk non-diabetic cases, DiCE changed-feature sets are
# generated under each model, and pairwise Jaccard similarity between the three models'
# changed-feature sets is averaged to a per-case consistency score. Produces a grayscale
# figure (per-case bars + distribution violins; this matches the actual plotted content --
# NOT a heatmap) and a summary table. dpi 600, PNG+PDF, no titles/captions.

import warnings, itertools, signal
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import dice_ml
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
class _Timeout(Exception):
    pass
def _handler(signum, frame):
    raise _Timeout()
signal.signal(signal.SIGALRM, _handler)
PER_CASE_TIMEOUT = 6
SEED = 42; np.random.seed(SEED)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data"; TAB = ROOT / "results" / "tables"; FIG = ROOT / "results" / "figures"

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "pdf.fonttype": 42,
                     "ps.fonttype": 42, "font.size": 11, "axes.edgecolor": "0.2",
                     "axes.linewidth": 0.8, "grid.color": "0.85"})
def savefig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight"); plt.close(fig)

PRED = ["age", "HE_BMI", "HE_wc", "HE_wt", "HE_ht", "N_EN", "N_CHO", "N_SUGAR",
        "N_CHOL", "sm_presnt", "dr_month", "pa_aerobic", "L_BR_FQ", "edu", "ho_incm", "D_1_1"]
CATEG = ["sm_presnt", "dr_month", "pa_aerobic"]; CONT = [c for c in PRED if c not in CATEG]
IMMUTABLE = ["age", "HE_ht", "edu", "ho_incm", "D_1_1"]; VARY = [c for c in PRED if c not in IMMUTABLE]
WAIST_CAP = 15.0; PCT_CAP = {"HE_BMI": 0.20, "HE_wt": 0.20}; N_CASES = 10

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 1.0

def make_models(X, y):
    return {"XGBoost": xgb.XGBClassifier(max_depth=4, n_estimators=300, learning_rate=0.05,
                                         eval_metric="logloss", random_state=SEED, n_jobs=4).fit(X, y),
            "CatBoost": CatBoostClassifier(depth=4, iterations=300, learning_rate=0.05,
                                           random_state=SEED, verbose=0).fit(X, y),
            "LightGBM": LGBMClassifier(max_depth=4, n_estimators=300, learning_rate=0.05,
                                       random_state=SEED, n_jobs=4, verbose=-1).fit(X, y)}

per_case = []
for sx in ["Male", "Female"]:
    full = pd.read_parquet(DATA / f"scored_{sx.lower()}.parquet")
    X = full[PRED].astype(float).values; y = full["diabetes"].values
    models = make_models(X, y)
    data_df = full[PRED + ["diabetes"]].copy()
    for c in CATEG:
        data_df[c] = data_df[c].astype(int).astype(str)
    hr = pd.read_parquet(DATA / f"highrisk_nondiab_{sx.lower()}.parquet")
    samp = hr.sample(min(N_CASES, len(hr)), random_state=SEED).reset_index(drop=True)
    explainers = {}
    for name, mdl in models.items():
        class Wrap:
            def __init__(self, mm): self.m = mm
            def predict_proba(self, Z):
                W = Z.copy()
                for c in CATEG:
                    if W[c].dtype == object:
                        W[c] = W[c].astype(float)
                return self.m.predict_proba(W[PRED].astype(float))
            def predict(self, Z):
                return (self.predict_proba(Z)[:, 1] >= 0.5).astype(int)
        d = dice_ml.Data(dataframe=data_df, continuous_features=CONT, outcome_name="diabetes")
        m = dice_ml.Model(model=Wrap(mdl), backend="sklearn", model_type="classifier")
        explainers[name] = dice_ml.Dice(d, m, method="random")
    for i in range(len(samp)):
        x = samp.iloc[i]
        row = samp.iloc[[i]][PRED].copy()
        for c in CATEG:
            row[c] = row[c].astype(int).astype(str)
        pr = {"HE_wc": [float(x.HE_wc - WAIST_CAP), float(x.HE_wc)]}
        for v, cap in PCT_CAP.items():
            pr[v] = [float(x[v] * (1 - cap)), float(x[v])]
        changed = {}
        for name, ex in explainers.items():
            try:
                signal.alarm(PER_CASE_TIMEOUT)
                cf = ex.generate_counterfactuals(row, total_CFs=4, desired_class=0,
                                                 features_to_vary=VARY, permitted_range=pr, random_seed=SEED)
                signal.alarm(0)
                cfdf = cf.cf_examples_list[0].final_cfs_df
                best = cfdf[PRED].astype(float).iloc[0]
                changed[name] = set(v for v in VARY if abs(float(best[v]) - float(x[v])) > 1e-6)
            except (_Timeout, Exception):
                signal.alarm(0)
                changed[name] = set()
        js = [jaccard(changed[a], changed[b]) for a, b in itertools.combinations(models, 2)]
        per_case.append([sx, i, round(float(np.mean(js)), 3)])

pc = pd.DataFrame(per_case, columns=["Sex", "case", "consistency"])
summary = pc.groupby("Sex").consistency.agg(["mean", "std", "min", "max"]).round(3).reset_index()
pc.to_csv(TAB / "table_robustness_percase.csv", index=False)
summary.to_csv(TAB / "table_robustness_summary.csv", index=False)
print("Inter-model consistency (mean pairwise Jaccard across XGB/CatBoost/LightGBM):")
print(summary.to_string(index=False))

# ---- Figure: (A) per-case bars both sexes; (B) violin distribution both sexes ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
ax = axes[0]
w = 0.4
for off, sx, c, h in [(-0.2, "Male", "#3b3b3b", ""), (0.2, "Female", "#a9a9a9", "///")]:
    sub = pc[pc.Sex == sx].reset_index(drop=True)
    ax.bar(sub["case"] + off, sub["consistency"], width=w, color=c, edgecolor="black",
           linewidth=0.5, hatch=h, label=sx)
    ax.axhline(summary[summary.Sex == sx]["mean"].values[0], color=c, ls="--", lw=1)
ax.set_xlabel("Case index"); ax.set_ylabel("Mean pairwise Jaccard consistency")
ax.set_xticks(range(N_CASES)); ax.legend(frameon=False)
ax.text(0.04, 0.92, "(A)", transform=ax.transAxes, fontsize=10, fontweight="bold")
ax = axes[1]
sns.violinplot(data=pc, x="Sex", y="consistency", palette={"Male": "#3b3b3b", "Female": "#a9a9a9"},
               inner="point", ax=ax, cut=0)
ax.set_xlabel(""); ax.set_ylabel("Consistency score")
ax.text(0.04, 0.92, "(B)", transform=ax.transAxes, fontsize=10, fontweight="bold")
savefig(fig, "fig_robustness")
print("Saved robustness tables and fig_robustness (bars + violins).")


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:05<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  2.59it/s]

100%|██████████| 1/1 [00:00<00:00,  2.58it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  2.05it/s]

100%|██████████| 1/1 [00:00<00:00,  2.05it/s]

Inter-model consistency (mean pairwise Jaccard across XGB/CatBoost/LightGBM):
   Sex  mean   std  min   max
Female 0.422 0.434  0.0 1.000
  Male 0.128 0.159  0.0 0.444


Saved robustness tables and fig_robustness (bars + violins).
